In [1]:
pip install pandas numpy scikit-learn matplotlib seaborn streamlit openpyxl


In [2]:
# Cell 1 — imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle
import os

# Helper: safe save path
OUTDIR = "."


In [3]:
# Cell 2 — Load & clean Kidney dataset
kidney_path = "kidney_disease - kidney_disease.csv"
kidney = pd.read_csv(kidney_path, na_values=["?"," "])
# quick cleanup: drop entirely empty columns
kidney = kidney.dropna(axis=1, how="all")
# convert numeric columns where possible
for c in kidney.columns:
    try:
        kidney[c] = pd.to_numeric(kidney[c])
    except:
        pass
# target column name variants: try to find classification/label
print("Kidney columns:", kidney.columns.tolist())
kidney = kidney.rename(columns={col:col.strip() for col in kidney.columns})
# drop rows where classification missing
if "classification" in kidney.columns:
    kidney = kidney.dropna(subset=["classification"])
else:
    # if target different name, try 'class' or 'status'
    print("Check kidney dataset target column: expected 'classification'")
kidney['classification'] = kidney['classification'].astype(str).str.strip().replace({'ckd':'ckd','notckd':'notckd'})
kidney['classification'] = kidney['classification'].map({'ckd':1,'notckd':0})
# fill remaining numeric NaN with median
num_cols = kidney.select_dtypes(include=[np.number]).columns.tolist()
kidney[num_cols] = kidney[num_cols].fillna(kidney[num_cols].median())
print("Kidney shape:", kidney.shape)


Kidney columns: ['id', 'age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'classification']
Kidney shape: (400, 26)


In [4]:
# Cell 3 — Train Kidney model
X_k = kidney.drop(columns=["classification"], errors='ignore')
y_k = kidney["classification"].astype(int)

# if any non-numeric columns remain, encode
for c in X_k.select_dtypes(include=['object']).columns:
    X_k[c] = LabelEncoder().fit_transform(X_k[c].astype(str))

# scale numeric if needed
scaler_k = StandardScaler()
Xk_num = X_k.select_dtypes(include=[np.number]).columns
X_k[Xk_num] = scaler_k.fit_transform(X_k[Xk_num])

X_train, X_test, y_train, y_test = train_test_split(X_k, y_k, test_size=0.2, random_state=42, stratify=y_k)
model_kidney = RandomForestClassifier(n_estimators=200, random_state=42)
model_kidney.fit(X_train, y_train)
y_pred = model_kidney.predict(X_test)
print("Kidney Accuracy:", accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))

# Save model + scaler + feature list
pickle.dump(model_kidney, open(os.path.join(OUTDIR,"kidney_model.pkl"),"wb"))
pickle.dump(scaler_k, open(os.path.join(OUTDIR,"kidney_scaler.pkl"),"wb"))
pickle.dump(X_k.columns.tolist(), open(os.path.join(OUTDIR,"kidney_features.pkl"),"wb"))


Kidney Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        30
           1       1.00      1.00      1.00        50

    accuracy                           1.00        80
   macro avg       1.00      1.00      1.00        80
weighted avg       1.00      1.00      1.00        80



In [5]:
# Cell 4 — Load & clean Liver dataset
liver_path = "indian_liver_patient - indian_liver_patient.csv"
liver = pd.read_csv(liver_path)
liver.columns = [c.strip() for c in liver.columns]
print("Liver columns:", liver.columns.tolist())
# target often 'Dataset' or 'Liver_disease' -> inspect and rename
# Many liver datasets use 'Dataset' with 1=patient, 2=not patient — we'll map to 1/0
if 'Dataset' in liver.columns:
    liver = liver.rename(columns={'Dataset':'Target'})
elif 'Liver_disease' in liver.columns:
    liver = liver.rename(columns={'Liver_disease':'Target'})
# convert and clean
for c in liver.columns:
    try:
        liver[c] = pd.to_numeric(liver[c])
    except:
        pass
# Map target to 0/1 if needed
if liver['Target'].nunique() > 2:
    # convert values like 1/2 to 1/0
    liver['Target'] = liver['Target'].map(lambda x: 1 if x==1 else 0)
# handle missing
liver = liver.dropna(axis=0, subset=['Target'])
num_cols = liver.select_dtypes(include=[np.number]).columns.tolist()
liver[num_cols] = liver[num_cols].fillna(liver[num_cols].median())
print("Liver shape:", liver.shape)


Liver columns: ['Age', 'Gender', 'Total_Bilirubin', 'Direct_Bilirubin', 'Alkaline_Phosphotase', 'Alamine_Aminotransferase', 'Aspartate_Aminotransferase', 'Total_Protiens', 'Albumin', 'Albumin_and_Globulin_Ratio', 'Dataset']
Liver shape: (583, 11)


In [6]:
# Cell 5 — Train Liver model
X_l = liver.drop(columns=['Target'], errors='ignore')
y_l = liver['Target'].astype(int)

# encode non-numeric if any
for c in X_l.select_dtypes(include=['object']).columns:
    X_l[c] = LabelEncoder().fit_transform(X_l[c].astype(str))

scaler_l = StandardScaler()
numcols_l = X_l.select_dtypes(include=[np.number]).columns
X_l[numcols_l] = scaler_l.fit_transform(X_l[numcols_l])

X_train, X_test, y_train, y_test = train_test_split(X_l, y_l, test_size=0.2, random_state=42, stratify=y_l)
model_liver = RandomForestClassifier(n_estimators=200, random_state=42)
model_liver.fit(X_train, y_train)
y_pred = model_liver.predict(X_test)
print("Liver Accuracy:", accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))

pickle.dump(model_liver, open(os.path.join(OUTDIR,"liver_model.pkl"),"wb"))
pickle.dump(scaler_l, open(os.path.join(OUTDIR,"liver_scaler.pkl"),"wb"))
pickle.dump(X_l.columns.tolist(), open(os.path.join(OUTDIR,"liver_features.pkl"),"wb"))


Liver Accuracy: 0.5982905982905983
              precision    recall  f1-score   support

           1       0.70      0.77      0.73        83
           2       0.24      0.18      0.20        34

    accuracy                           0.60       117
   macro avg       0.47      0.47      0.47       117
weighted avg       0.56      0.60      0.58       117



In [7]:
# Cell 6 — Load & clean Parkinsons dataset
parkinsons_path = "parkinsons - parkinsons.csv"
par = pd.read_csv(parkinsons_path)
par.columns = [c.strip() for c in par.columns]
print("Parkinsons columns sample:", par.columns[:10].tolist())
# target usually 'status' (0/1)
if 'status' in par.columns:
    par['status'] = par['status'].astype(int)
else:
    # try 'status ' etc.
    pass
# fill na numeric
par_num = par.select_dtypes(include=[np.number]).columns
par[par_num] = par[par_num].fillna(par[par_num].median())
print("Parkinsons shape:", par.shape)


Parkinsons columns sample: ['name', 'MDVP:Fo(Hz)', 'MDVP:Fhi(Hz)', 'MDVP:Flo(Hz)', 'MDVP:Jitter(%)', 'MDVP:Jitter(Abs)', 'MDVP:RAP', 'MDVP:PPQ', 'Jitter:DDP', 'MDVP:Shimmer']
Parkinsons shape: (195, 24)


In [8]:
# Cell 7 — Train Parkinsons model
X_p = par.drop(columns=['status'], errors='ignore')
y_p = par['status'].astype(int)

# encode non-numeric if any
for c in X_p.select_dtypes(include=['object']).columns:
    X_p[c] = LabelEncoder().fit_transform(X_p[c].astype(str))

scaler_p = StandardScaler()
numcols_p = X_p.select_dtypes(include=[np.number]).columns
X_p[numcols_p] = scaler_p.fit_transform(X_p[numcols_p])

X_train, X_test, y_train, y_test = train_test_split(X_p, y_p, test_size=0.2, random_state=42, stratify=y_p)
model_parkinson = RandomForestClassifier(n_estimators=200, random_state=42)
model_parkinson.fit(X_train, y_train)
y_pred = model_parkinson.predict(X_test)
print("Parkinsons Accuracy:", accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))

pickle.dump(model_parkinson, open(os.path.join(OUTDIR,"parkinson_model.pkl"),"wb"))
pickle.dump(scaler_p, open(os.path.join(OUTDIR,"parkinson_scaler.pkl"),"wb"))
pickle.dump(X_p.columns.tolist(), open(os.path.join(OUTDIR,"parkinson_features.pkl"),"wb"))


Parkinsons Accuracy: 0.9743589743589743
              precision    recall  f1-score   support

           0       0.91      1.00      0.95        10
           1       1.00      0.97      0.98        29

    accuracy                           0.97        39
   macro avg       0.95      0.98      0.97        39
weighted avg       0.98      0.97      0.97        39



In [9]:
# Cell 8 — Create requirements.txt (optional)
reqs = """pandas
numpy
scikit-learn
matplotlib
seaborn
streamlit
openpyxl
"""
with open("requirements.txt","w") as f:
    f.write(reqs)
print("Saved requirements.txt")


Saved requirements.txt


In [13]:
arr = np.array([inputs[f] for f in parkinson_features if f!="status"]).reshape(1, -1)


In [14]:
import streamlit as st
st.set_page_config(page_title="Multiple Disease Prediction", layout="wide")


2025-11-29 12:31:34.718 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
